In [ ]:
import random
import sys
from datasets import load_dataset, get_dataset_config_names
from typing import List, Dict, Any

# 🧑‍🎓 오늘의 미션: AI가 사람의 선호를 어떻게 배우는지 탐험하기!
# 💡 데이터셋 설명: nayohan/SentimentSynth-ko
# 이 데이터셋은 'DPO (Direct Preference Optimization)'라는 아주 중요한 학습 방식을 위해 만들어졌어요.
# 핵심은 "어떤 질문(prompt)에 대해 여러 답변이 있을 때, 사람이 '이 답변이 저 답변보다 훨씬 좋아!' 라고 골라주는 선호 데이터"입니다.
# 저희는 이 데이터를 가지고 AI가 어떤 답변을 '선호'하는지 패턴을 분석해 볼 거예요!

# --- ⚙️ 설정 변수 ---
DATASET_NAME = "nayohan/SentimentSynth-ko"
SAMPLE_COUNT = 5  # 초보자분들을 위해 일단 5개의 샘플만 살펴봅시다!
DATASET_SPLIT = "train"

# --------------------------------------------------------------------
# 🚀 1단계: 데이터셋 로드 준비 (스트리밍 모드를 먼저 시도해요!)
# --------------------------------------------------------------------

print("========================================================================")
print("✨ [튜터 모드] 데이터를 로드할 준비를 합니다. AI의 선호도를 분석하는 재미있는 여정을 떠날게요!")
print("========================================================================\n")

dataset = None
sample_data_list = []

# 1. 설정 가능한 Config 확인
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 예: 첫 번째 config로 로드
    selected_config = configs[0]
except Exception as e:
    print(f"ℹ️ 해당 데이터셋은 별도의 Config가 없거나 기본(default) 설정만 제공됩니다. (에러: {e})")
    selected_config = None


# 2. 스트리밍 로드 시도 (가장 빠르고 효율적!)
try:
    print(f"⚡️ [Attempt 1/2] 데이터셋을 스트리밍(Streaming) 모드로 로드 시도 중입니다... (매우 빠를 거예요!)")
    # streaming=True를 사용하여 대용량 데이터셋을 메모리에 다 올리지 않고 흘러가듯 처리해요.
    dataset = load_dataset(DATASET_NAME, name=selected_config, split=DATASET_SPLIT, streaming=True)
    print("🎉 성공! 스트리밍 모드로 데이터셋 로드에 성공했어요. 다음 단계로 넘어갑니다.")

except Exception as e:
    print(f"\n⚠️ [주의] 스트리밍 로드 실패! ({e}) 이 데이터셋은 스트리밍 모드에서 문제가 발생했어요.")
    print("🔄 대안: 소량의 샘플만 다운로드하여 일반 Dataset으로 다시 시도합니다.")
    
    try:
        # 실패 시, 소량만 다운로드하여 일반 Dataset으로 대체
        dataset = load_dataset(DATASET_NAME, name=selected_config, split=DATASET_SPLIT, streaming=False)
        print("✅ 성공적으로 일반 Dataset 모드로 로드했습니다. 이제 안전하게 분석할 수 있어요!")
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드 실패. 모든 시도에도 불구하고 데이터 로드에 실패했습니다. (에러: {e_fallback})")
        sys.exit()


# 3. 실제로 분석할 샘플 데이터 준비 (테이크 & 리스트 변환)
print("\n========================================================================")
print(f"🔬 2단계: 분석할 {SAMPLE_COUNT}개의 샘플을 추출하고 준비합니다.")
print("========================================================================")

# streaming 모드와 일반 모드 모두에 안전한 방식으로 데이터를 샘플링해요!
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    sample_data_iterator = dataset.take(SAMPLE_COUNT)
    # iter()를 사용하여 순회할 준비를 합니다.
    sample_data_list = []
    for i, sample in enumerate(sample_data_iterator):
        sample_data_list.append(sample)
else:
    # 일반 데이터셋 (Dataset)인 경우
    sample_data_list = list(dataset.take(SAMPLE_COUNT))

print(f"🔍 {len(sample_data_list)}개의 샘플을 준비하여 분석을 시작합니다!")

# --------------------------------------------------------------------
# 🌟 3단계: 창의적 분석 - AI의 '최선의 답변'를 추론하는 코딩 실습!
# --------------------------------------------------------------------

print("\n\n==================================================================================================")
print("🏆 3단계: [AI 선호도 분석] '왜 chosen가 더 좋은지' 추론해 봅시다!")
print("==================================================================================================")

def analyze_preference(sample: Dict[str, str]):
    """
    주어진 Prompt와 chosen/rejected 답변을 비교하여 선호 패턴을 분석하고 튜터링합니다.
    """
    prompt = sample.get('prompt', 'N/A')
    chosen = sample.get('chosen', 'N/A')
    rejected = sample.get('rejected', 'N/A')
    
    print("\n" + "="*80)
    print(f"🤖 ✏️ Prompt 분석 시작 (Context): {prompt[:60]}...")
    
    print("\n\n>>> 🧠 모델의 평가 (AI Tutor의 관점) <<<")
    
    # 1. Prompt 분석 (데이터 특성 이해)
    print(f"\n⭐ [핵심 목표] 이 프롬프트가 요구하는 것은 '{prompt[:30]}...'입니다.")
    print("   -> AI는 이 질문의 의도를 정확히 파악해야 해요!")
    
    # 2. Chosen vs Rejected 비교 (실제 학습 내용)
    print("-" * 30 + " ✨ Chosen (선호됨) " + "-" * 25)
    print(f"    [🌟 내용]: {chosen[:80]}...")
    print("    [💡 Tuber Tip]: 이 답변은 명확하고, 요청된 정보를 모두 포함하며, 어조가 자연스러워요.")
    
    print("\n" + "-" * 30 + " 🧱 Rejected (덜 선호됨) " + "-" * 25)
    print(f"    [🗑️ 내용]: {rejected[:80]}...")
    print("    [📉 Tuber Tip]: 이 답변은 너무 짧거나, 주제에서 벗어났거나, 핵심 정보가 빠져 있을 수 있어요.")

    # 3. 결론 도출 (가장 중요한 학습 결과)
    print("\n🏆 [최종 결론]: AI는 'chosen' 답변의 다음 특징을 학습했습니다!")
    if "보다" in chosen and "직접" in chosen:
        print("✨ 1. [정보 완전성]: 'chosen'는 단순히 내용을 나열하는 것이 아니라, 비교나 부연 설명을 통해 답변의 깊이를 높였어요.")
    elif len(chosen.split()) > len(rejected.split()):
        print("✨ 2. [길이와 상세도]: 답변이 더 길고 상세할수록, AI는 더 많은 정보를 제공해야 좋은 답변이라고 인식합니다.")
    elif "좋습니다" in chosen or "좋아요" in chosen:
        print("✨ 3. [사용자 친화적 어조]: 'chosen'는 사용자에게 공감하고 친절하게 마무리 짓는 '어조(Tone)'를 사용했습니다.")
    else:
        print("✨ 4. [일관성]: 'chosen'는 Prompt의 주제에서 벗어나지 않고, 일관성 있게 답하는 능력을 보여줍니다.")


# 🚀 루프를 돌면서 분석 실행
for i, sample in enumerate(sample_data_list):
    print("\n" + "#" * 100)
    print(f"🌐 {i+1}번째 샘플 분석 중... (Sample {i+1}/{len(sample_data_list)})")
    analyze_preference(sample)

print("\n==================================================================================================")
print("👏 축하합니다! 훌륭하게 데이터셋의 패턴을 분석했습니다. 😊")
print("이제 여러분은 AI가 어떤 답변을 '좋게' 평가하는지, 그 비결을 이해하게 되었어요!")
print("다음엔 이 분석을 바탕으로 직접 나만의 질문/답변 쌍을 만들어 볼 수 있겠죠? 화이팅!")
print("==================================================================================================")